# Quality Signal (theta)

**Purpose:** Load the dataset and validate the quality signal theta (PROF).

PROF is pre-calculated in the prof CSV files using the Novy-Marx & Medhat (2025) definition:

$$\theta_{i,t} = \frac{\text{Rev} - \text{COGS} - (\text{SGA} - \text{R\&D}) - \text{Int}}{\text{BE}_{t-1} + \text{MIB}_{t-1}}$$

We read PROF directly and use it as theta without modification.

**Output:** `data/processed/theta.parquet` — one row per (Ticker, Year) with theta_{i,t}.

In [2]:
import sys
from pathlib import Path
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

notebook_dir = Path().resolve()
sys.path.insert(0, str(notebook_dir.parents[0] / "src"))

from config import PROF_DIR, ACC_DIR
from data_loader import load_all, describe_dataset

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 40)

## 1. Load data

In [3]:
data = load_all()
describe_dataset(data)

11:47:09  WARNING  Tickers in prof but NOT in acc (2): ['QLIRO.ST', 'REBL.HE']
11:47:09  INFO  Loading 633 tickers present in both directories.
11:47:11  WARNING    20202.OL/acc: dropping 1 rows with missing required columns ['Year', 'Ticker', 'ACT', 'CHE', 'LCT', 'AT', 'OANCF', 'PPEGT']
11:47:12  WARNING    AAB.CO/prof: dropping 1 rows with missing required columns ['Year', 'Ticker', 'PROF']
11:47:12  WARNING    AAB.CO/acc: dropping 8 rows with missing required columns ['Year', 'Ticker', 'ACT', 'CHE', 'LCT', 'AT', 'OANCF', 'PPEGT']
11:47:13  WARNING    AAK.ST/prof: dropping 1 rows with missing required columns ['Year', 'Ticker', 'PROF']
11:47:13  WARNING    AAK.ST/acc: dropping 5 rows with missing required columns ['Year', 'Ticker', 'ACT', 'CHE', 'LCT', 'AT', 'OANCF', 'PPEGT']
11:47:14  WARNING    ABB.ST/prof: dropping 1 rows with missing required columns ['Year', 'Ticker', 'PROF']
11:47:15  WARNING    ABB.ST/acc: dropping 16 rows with missing required columns ['Year', 'Ticker', 'ACT'

ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

## 2. Extract theta

PROF is used directly as theta. We keep metadata columns for later use.

In [ ]:
META_COLS  = ['Ticker', 'Year', 'CompanyName', 'Industry', 'Sector']
META_COLS  = [c for c in META_COLS if c in data.columns]

theta = data[META_COLS + ['PROF']].copy()
theta = theta.rename(columns={'PROF': 'theta'})
theta = theta.dropna(subset=['theta'])
theta = theta.sort_values(['Ticker', 'Year']).reset_index(drop=True)

print(f'Firm-years with valid theta: {len(theta):,}')
print(f'Tickers: {theta["Ticker"].nunique():,}')
print(f'Year range: {theta["Year"].min()} – {theta["Year"].max()}')
theta.head(10)

## 3. Descriptive statistics

In [ ]:
print('Overall theta distribution:')
print(theta['theta'].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]))

In [ ]:
# Cross-sectional statistics per year
annual_stats = (
    theta.groupby('Year')['theta']
    .agg(['mean', 'median', 'std', 'count'])
    .round(4)
)
print('Annual cross-sectional statistics:')
annual_stats

## 4. Distribution plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Overall distribution
axes[0].hist(theta['theta'].dropna(), bins=80, edgecolor='none', color='steelblue', alpha=0.8)
axes[0].set_title('Distribution of theta (all firm-years)')
axes[0].set_xlabel('theta')
axes[0].set_ylabel('Count')
axes[0].axvline(theta['theta'].median(), color='firebrick', lw=1.5, linestyle='--', label='Median')
axes[0].legend()

# Annual mean and +/- 1 std
ax = axes[1]
years = annual_stats.index
ax.plot(years, annual_stats['mean'], marker='o', ms=4, color='steelblue', label='Mean')
ax.fill_between(
    years,
    annual_stats['mean'] - annual_stats['std'],
    annual_stats['mean'] + annual_stats['std'],
    alpha=0.15, color='steelblue', label='±1 std'
)
ax.set_title('Annual mean theta ± 1 std')
ax.set_xlabel('Year')
ax.set_ylabel('theta')
ax.legend()
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig('../output/theta_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. OPTION: Winsorization

Winsorization is **disabled by default**. Set `WINSORIZE = True` to activate.

Cross-sectional winsorization at 1st/99th percentile per year, as per methodology.

In [ ]:
WINSORIZE = False   # <-- set True to enable
WINS_LOW  = 0.01
WINS_HIGH = 0.99

if WINSORIZE:
    def _winsorize_year(group):
        lo = group['theta'].quantile(WINS_LOW)
        hi = group['theta'].quantile(WINS_HIGH)
        group = group.copy()
        group['theta'] = group['theta'].clip(lower=lo, upper=hi)
        return group

    theta = theta.groupby('Year', group_keys=False).apply(_winsorize_year)
    print(f'Winsorization applied at [{WINS_LOW:.0%}, {WINS_HIGH:.0%}] per year.')
else:
    print('Winsorization not applied.')

## 6. Save output

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

theta.to_parquet('../data/processed/theta.parquet', index=False)
# Also save full merged dataset for downstream notebooks
data.to_parquet('../data/processed/data_merged.parquet', index=False)

print('Saved:')
print('  data/processed/theta.parquet')
print('  data/processed/data_merged.parquet')
print(f'  {len(theta):,} firm-years, {theta["Ticker"].nunique()} tickers')